# 3-3. FastAPI 예측 API 계약 확인 실습

이 notebook은 예측 API가 `score`를 반환한다는 사실만 확인하지 않고, 요청/응답 계약이 AI 서비스 품질 추적에 필요한 증거를 담는지 점검하는 실습입니다. 실제 FastAPI 서버는 로컬 Lab 또는 외부 제공 API에서 실행될 수 있고, JupyterLite에서는 같은 입출력 구조를 브라우저 안에서 재현합니다.

핵심 흐름은 `실행 경로 고정 -> payload 데이터 brief -> 정상 응답 계약 확인 -> OpenAPI schema 확인 -> 오류 응답 분리 -> deployment artifact 경계 확인 -> 4장 로그 handoff`입니다. FastAPI를 처음부터 구현하는 실습이 아니라, 제공된 API 계약을 QA 관점에서 읽고 호출 결과를 판단하는 실습입니다.

| 받은 업무 | 현장 관측값 | 결정 압박 |
| --- | --- | --- |
| `/predict`가 release gate에 필요한 추적 필드를 반환하는지 확인합니다 | 2장에서 `model_version`, `threshold`, Precision/Recall, FP/FN 기준이 정리되었습니다 | API 응답이 2장 평가 기준과 연결되지 않으면 운영 로그와 지표를 해석하기 어렵습니다 |

| 이 notebook에서 만드는 증거 | 확인 대상 | 보고서 사용 |
| --- | --- | --- |
| payload field table, response field gate, OpenAPI schema table, 422 error detail, deployment boundary, serving handoff sentence | Lite fallback 또는 외부 API 응답, prepared Docker/Kubernetes artifact | API 계약 승인/보류 의견, 4장 observability 확인 항목 |

## 1. 실행 경로와 API 확인 범위

### 1-1. Lite fallback과 외부 API 경로 구분

이 셀의 판단은 지금 API 응답을 어디서 얻는지 명확히 하는 것입니다. 강의장에서는 외부 API가 제공될 수 있고, 로컬 일반 환경에서는 FastAPI 앱을 직접 띄울 수 있으며, JupyterLite에서는 서버를 실행하지 않고 같은 계약을 모사합니다.

실행 경로가 달라도 QA 질문은 같습니다. 요청 payload가 학습 feature와 맞는지, 정상 응답이 `request_id`, `model_version`, `score`, `threshold`, `prediction`을 담는지, 잘못된 입력이 422 검증 오류로 분리되는지 확인합니다. 외부 API가 제공되면 다음 코드 셀의 `EXTERNAL_API_BASE_URL`에 주소를 직접 입력하거나 로컬 실행에서 `CH03_API_BASE_URL` 환경변수를 지정합니다.

이 단계에서는 아직 API를 호출하지 않습니다. 먼저 helper 준비, 실행 mode, 외부 API 설정 여부, Lite fallback 사용 여부를 표로 남깁니다.

In [1]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.request
from typing import Any

import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
lite = prepared.aiq_lite

for name in runtime.LITE_NAMES:
    globals()[name] = getattr(lite, name)

EXTERNAL_API_BASE_URL = ""  # 예: "https://api.example.com"; 비워 두면 Lite fallback을 사용합니다.
EXTERNAL_API_BASE_URL = (EXTERNAL_API_BASE_URL or os.environ.get("CH03_API_BASE_URL", "")).rstrip("/")

runtime_contract = pd.DataFrame(
    [
        {"item": "package_module", "value": lite.__name__, "qa_use": "Lite 또는 local helper 기준을 확인합니다."},
        {"item": "external_api_base_url", "value": EXTERNAL_API_BASE_URL or "not_set", "qa_use": "외부 API가 지정되면 `/predict`로 실제 호출을 시도합니다."},
        {"item": "fallback_mode", "value": "lite_contract_simulation", "qa_use": "외부 API가 없거나 실패하면 같은 계약의 fallback 응답으로 실습을 진행합니다."},
        {"item": "required_response_fields", "value": "request_id, model_version, score, threshold, prediction", "qa_use": "운영 추적에 필요한 최소 응답 필드입니다."},
    ]
)

display(runtime_contract)

,item,value,qa_use
0,package_module,ai_quality.lite,Lite 또는 local helper 기준을 확인합니다.
1,external_api_base_url,not_set,외부 API가 지정되면 `/predict`로 실제 호출을 시도합니다.
2,fallback_mode,lite_contract_simulation,외부 API가 없거나 실패하면 같은 계약의 fallback 응답으로 실습을 진행합니다.
3,required_response_fields,"request_id, model_version, score, threshold, p...",운영 추적에 필요한 최소 응답 필드입니다.


이 출력에서 확인할 핵심은 응답 출처를 보고서에서 구분해야 한다는 점입니다. 외부 API를 호출했다면 live API evidence이고, fallback을 썼다면 API 계약 구조를 재현한 browser evidence입니다.

## 2. 요청 payload를 데이터로 검토

### 2-1. payload source, row grain, feature 계약 확인

이 셀의 판단은 API 요청도 모델 입력 데이터라는 점을 확인하는 것입니다. `/predict` 요청 한 건은 모델이 score를 계산할 sample 한 row와 같으며, 이 row의 feature 이름과 타입이 2장 평가 조건과 맞아야 합니다.

payload에는 `request_id`와 여섯 개 feature가 들어갑니다. `request_id`는 모델 입력이 아니라 추적 키이고, 나머지 feature는 score 생성에 사용되는 모델 입력입니다.

이 단계에서는 아직 응답을 보지 않습니다. 먼저 요청 데이터 자체가 충분한지 확인해야 정상 응답의 `score`와 `prediction`을 해석할 수 있습니다.

In [2]:
payload = serving_payload()

payload_brief = pd.DataFrame(
    [
        {"check": "payload_source", "observed": "serving_payload() sample", "qa_use": "실습용 정상 요청 출처를 남깁니다."},
        {"check": "row_grain", "observed": "one prediction request", "qa_use": "한 요청이 모델 입력 sample 하나임을 설명합니다."},
        {"check": "feature_count", "observed": len(FEATURE_COLUMNS), "qa_use": "학습 feature와 API feature 수를 비교합니다."},
        {"check": "request_id_present", "observed": "request_id" in payload, "qa_use": "응답과 로그 연결 가능성을 확인합니다."},
        {"check": "threshold_expected", "observed": THRESHOLD, "qa_use": "score를 prediction으로 바꿀 운영 기준입니다."},
    ]
)
payload_table = pd.DataFrame(
    [
        {
            "field": field,
            "value": payload.get(field),
            "role": "trace_key" if field == "request_id" else "model_feature",
            "python_type": type(payload.get(field)).__name__,
        }
        for field in ["request_id", *FEATURE_COLUMNS]
    ]
)
feature_contract = pd.DataFrame(
    [
        {
            "feature": feature,
            "in_payload": feature in payload,
            "numeric": isinstance(payload.get(feature), (int, float)),
            "value": payload.get(feature),
            "qa_status": "pass" if feature in payload and isinstance(payload.get(feature), (int, float)) else "hold",
        }
        for feature in FEATURE_COLUMNS
    ]
)

display(payload_brief)
display(payload_table)
display(feature_contract)

,check,observed,qa_use
0,payload_source,serving_payload() sample,실습용 정상 요청 출처를 남깁니다.
1,row_grain,one prediction request,한 요청이 모델 입력 sample 하나임을 설명합니다.
2,feature_count,6,학습 feature와 API feature 수를 비교합니다.
3,request_id_present,True,응답과 로그 연결 가능성을 확인합니다.
4,threshold_expected,0.5,score를 prediction으로 바꿀 운영 기준입니다.


,field,value,role,python_type
0,request_id,lab-03-valid-001,trace_key,str
1,heart_rate,118,model_feature,int
2,respiratory_rate,25,model_feature,int
3,body_temperature,37.9,model_feature,float
4,oxygen_saturation,92,model_feature,int
5,systolic_blood_pressure,146,model_feature,int
6,diastolic_blood_pressure,92,model_feature,int


,feature,in_payload,numeric,value,qa_status
0,heart_rate,True,True,118.0,pass
1,respiratory_rate,True,True,25.0,pass
2,body_temperature,True,True,37.9,pass
3,oxygen_saturation,True,True,92.0,pass
4,systolic_blood_pressure,True,True,146.0,pass
5,diastolic_blood_pressure,True,True,92.0,pass


이 출력에서 확인할 핵심은 정상 요청이 score를 계산할 feature를 모두 갖고 있다는 점입니다. feature 누락이나 타입 오류가 있으면 모델 품질 문제가 아니라 입력 계약 문제로 먼저 분리해야 합니다.

`request_id`는 feature가 아닙니다. 이 값은 같은 요청의 응답, 로그, 지표를 연결하기 위한 키이며, 4장 observability에서 structured log를 추적할 때 다시 사용합니다.

### 2-2. 잘못된 payload를 미리 만들고 기대 실패를 정의

이 셀의 판단은 실패 입력을 의도적으로 정의해, 어떤 오류가 모델 추론 전에 차단되어야 하는지 정하는 것입니다. 오류 테스트는 실패를 만들기 위한 장난이 아니라 API 계약이 데이터를 보호하는지 확인하는 release gate입니다.

이번 invalid payload는 `heart_rate`에 문자열을 넣고 다른 필수 feature를 누락합니다. 기대 결과는 500 서버 오류가 아니라 422 검증 오류입니다.

이 기대를 먼저 써 두면 실제 오류 응답을 봤을 때 API 문제인지 입력 계약 문제인지 빠르게 분리할 수 있습니다.

In [3]:
invalid_payload = {
    "request_id": "lab-03-invalid-001",
    "heart_rate": "not-a-number",
}
expected_error_contract = pd.DataFrame(
    [
        {"input_issue": "heart_rate_type", "payload_value": invalid_payload["heart_rate"], "expected_response": 422, "qa_reason": "숫자 feature 타입 오류는 schema 검증에서 차단되어야 합니다."},
        {"input_issue": "missing_required_features", "payload_value": sorted(set(FEATURE_COLUMNS) - set(invalid_payload)), "expected_response": 422, "qa_reason": "필수 feature 누락은 예측 로직 전에 차단되어야 합니다."},
        {"input_issue": "request_id_present", "payload_value": invalid_payload["request_id"], "expected_response": "included_in_error_context", "qa_reason": "실패 요청도 추적 가능해야 합니다."},
    ]
)

display(expected_error_contract)

,input_issue,payload_value,expected_response,qa_reason
0,heart_rate_type,not-a-number,422,숫자 feature 타입 오류는 schema 검증에서 차단되어야 합니다.
1,missing_required_features,"[body_temperature, diastolic_blood_pressure, o...",422,필수 feature 누락은 예측 로직 전에 차단되어야 합니다.
2,request_id_present,lab-03-invalid-001,included_in_error_context,실패 요청도 추적 가능해야 합니다.


이 표에서 확인할 핵심은 오류 응답도 품질 증거라는 점입니다. 정상 응답만 확인하면 잘못된 입력이 모델 추론으로 들어가는지 알 수 없습니다.

## 3. 정상 응답 계약 확인

### 3-1. 외부 API 또는 Lite fallback으로 `/predict` 호출

이 셀의 판단은 정상 payload가 어떤 응답 계약을 반환하는지 확인하는 것입니다. 외부 API URL이 설정되어 있으면 실제 `/predict`로 호출하고, 설정이 없거나 호출에 실패하면 Lite fallback 응답으로 같은 필드 구조를 확인합니다.

정상 응답의 핵심은 `score` 숫자 하나가 아닙니다. `model_version`은 2장 평가 기록과 연결되고, `threshold`는 score가 prediction으로 바뀐 기준이며, `request_id`는 로그 추적 키입니다.

이 요청은 정답 label이 있는 평가 데이터가 아니라 API 계약 확인용 sample입니다. 따라서 응답 하나로 모델 품질을 판단하지 않고, 추적 필드와 변환 기준이 남는지 확인합니다.

In [4]:
def call_predict(payload_to_send: dict[str, object]) -> dict[str, object]:
    if EXTERNAL_API_BASE_URL:
        request = urllib.request.Request(
            f"{EXTERNAL_API_BASE_URL}/predict",
            data=json.dumps(payload_to_send).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        try:
            with urllib.request.urlopen(request, timeout=10) as response:
                return {
                    "source": "external_api",
                    "status_code": response.status,
                    "payload": json.loads(response.read().decode("utf-8")),
                    "fallback_reason": None,
                }
        except urllib.error.HTTPError as exc:
            return {
                "source": "external_api",
                "status_code": exc.code,
                "payload": json.loads(exc.read().decode("utf-8")),
                "fallback_reason": None,
            }
        except Exception as exc:
            fallback_response = post_predict(payload_to_send)
            return {
                "source": "lite_fallback_after_external_error",
                "status_code": fallback_response.status_code,
                "payload": fallback_response.json(),
                "fallback_reason": f"{type(exc).__name__}: {exc}",
            }

    fallback_response = post_predict(payload_to_send)
    return {
        "source": "lite_contract_simulation",
        "status_code": fallback_response.status_code,
        "payload": fallback_response.json(),
        "fallback_reason": None,
    }


valid_call = call_predict(payload)
valid_response_payload = valid_call["payload"]

response_source = pd.DataFrame(
    [
        {"item": "response_source", "value": valid_call["source"], "qa_use": "live API evidence인지 fallback evidence인지 구분합니다."},
        {"item": "status_code", "value": valid_call["status_code"], "qa_use": "정상 경로는 200이어야 합니다."},
        {"item": "fallback_reason", "value": valid_call["fallback_reason"] or "not_applicable", "qa_use": "외부 API 실패 시 fallback 사용 사유를 남깁니다."},
    ]
)
response_table = pd.DataFrame(
    [
        {"field": field, "value": valid_response_payload.get(field), "present": field in valid_response_payload}
        for field in ["request_id", "model_version", "score", "threshold", "prediction"]
    ]
)
response_gate = pd.DataFrame(
    [
        {"gate": "status_200", "observed": valid_call["status_code"] == 200, "qa_status": "pass" if valid_call["status_code"] == 200 else "hold"},
        {"gate": "required_fields_present", "observed": bool(response_table["present"].all()), "qa_status": "pass" if bool(response_table["present"].all()) else "hold"},
        {"gate": "request_id_round_trip", "observed": valid_response_payload.get("request_id") == payload.get("request_id"), "qa_status": "pass" if valid_response_payload.get("request_id") == payload.get("request_id") else "review"},
        {"gate": "threshold_matches_ch02", "observed": valid_response_payload.get("threshold") == THRESHOLD, "qa_status": "pass" if valid_response_payload.get("threshold") == THRESHOLD else "review"},
        {"gate": "prediction_label_allowed", "observed": valid_response_payload.get("prediction") in [POSITIVE_LABEL, NEGATIVE_LABEL], "qa_status": "pass" if valid_response_payload.get("prediction") in [POSITIVE_LABEL, NEGATIVE_LABEL] else "hold"},
    ]
)

display(response_source)
display(response_table)
display(response_gate)

,item,value,qa_use
0,response_source,lite_contract_simulation,live API evidence인지 fallback evidence인지 구분합니다.
1,status_code,200,정상 경로는 200이어야 합니다.
2,fallback_reason,not_applicable,외부 API 실패 시 fallback 사용 사유를 남깁니다.


,field,value,present
0,request_id,lab-03-valid-001,True
1,model_version,v1,True
2,score,0.3521,True
3,threshold,0.5,True
4,prediction,low_risk,True


,gate,observed,qa_status
0,status_200,True,pass
1,required_fields_present,True,pass
2,request_id_round_trip,True,pass
3,threshold_matches_ch02,True,pass
4,prediction_label_allowed,True,pass


이 출력에서 확인할 핵심은 정상 응답이 2장 평가 기준과 이어지는 필드를 갖고 있다는 점입니다. `threshold`가 2장 운영 기준과 다르면 같은 score라도 prediction이 달라질 수 있으므로 `review` 대상입니다.

응답 출처가 fallback이면 보고서에 live API를 호출했다고 쓰면 안 됩니다. 대신 “Lite fallback 계약에서 필드 구조를 확인했고, live API smoke test는 외부 API 제공 시 재확인 필요”라고 남깁니다.

### 3-2. score, threshold, prediction 관계 확인

이 셀의 판단은 응답의 `score`가 바로 최종 판단이 아니라 `threshold`를 거쳐 `prediction`으로 변환된다는 점을 확인하는 것입니다. QA는 score와 prediction을 같은 말로 쓰지 않습니다.

`score`는 positive class 가능성에 가까운 연속값이고, `threshold`는 class 변환 기준이며, `prediction`은 최종 반환 class입니다. 이 구분이 있어야 threshold 변경 효과와 모델 score 변화 효과를 분리할 수 있습니다.

이 표의 판단은 정상 응답 한 건에서 score, threshold, prediction 관계가 일관적인지 확인하는 것입니다.

In [5]:
score_value = float(valid_response_payload["score"])
threshold_value = float(valid_response_payload["threshold"])
expected_prediction = POSITIVE_LABEL if score_value >= threshold_value else NEGATIVE_LABEL

score_prediction_audit = pd.DataFrame(
    [
        {"step": "score", "observed": round(score_value, 4), "qa_meaning": "prediction 이전의 연속값입니다."},
        {"step": "threshold", "observed": threshold_value, "qa_meaning": "class 변환 기준입니다."},
        {"step": "expected_prediction", "observed": expected_prediction, "qa_meaning": "score와 threshold로 계산한 기대 class입니다."},
        {"step": "response_prediction", "observed": valid_response_payload["prediction"], "qa_meaning": "API가 실제 반환한 class입니다."},
        {"step": "consistency", "observed": expected_prediction == valid_response_payload["prediction"], "qa_meaning": "응답 내부 변환 기준 일치 여부입니다."},
    ]
)

display(score_prediction_audit)

,step,observed,qa_meaning
0,score,0.3521,prediction 이전의 연속값입니다.
1,threshold,0.5,class 변환 기준입니다.
2,expected_prediction,low_risk,score와 threshold로 계산한 기대 class입니다.
3,response_prediction,low_risk,API가 실제 반환한 class입니다.
4,consistency,True,응답 내부 변환 기준 일치 여부입니다.


이 출력에서 확인할 핵심은 score와 prediction을 분리해서 기록해야 한다는 점입니다. 4장에서 score distribution과 prediction distribution을 볼 때도 이 구분이 유지되어야 합니다.

## 4. OpenAPI schema와 실제 응답 비교

### 4-1. 문서화된 요청/응답 계약 확인

이 셀의 판단은 OpenAPI schema가 실제 테스트 케이스의 기준으로 충분한지 확인하는 것입니다. API 문서가 있어야 client, QA, 운영 담당자가 같은 필드 이름과 오류 구조를 공유할 수 있습니다.

OpenAPI schema는 테스트를 대체하지 않습니다. schema가 있다고 해서 실제 응답이 항상 그 구조를 따르는 것은 아니므로, 정상 호출 결과와 함께 비교해야 합니다.

여기서는 `PredictionPayload`, `PredictionOutput`, `HTTPValidationError`가 노출되는지 확인합니다.

In [6]:
openapi = openapi_contract()
schemas = openapi["components"]["schemas"]

schema_table = pd.DataFrame(
    [
        {"schema": schema_name, "present": schema_name in schemas, "qa_use": qa_use}
        for schema_name, qa_use in [
            ("PredictionPayload", "요청 feature와 request_id 계약"),
            ("PredictionOutput", "응답 추적 필드 계약"),
            ("HTTPValidationError", "검증 오류 응답 계약"),
        ]
    ]
)
payload_schema = schemas["PredictionPayload"]
payload_required = set(payload_schema.get("required", []))
payload_schema_table = pd.DataFrame(
    [
        {
            "field": field,
            "required": field in payload_required,
            "documented_type": payload_schema["properties"].get(field, {}).get("type"),
            "training_feature": field in FEATURE_COLUMNS,
            "qa_status": "pass" if field in payload_schema["properties"] else "hold",
        }
        for field in ["request_id", *FEATURE_COLUMNS]
    ]
)
output_required = set(schemas["PredictionOutput"].get("required", []))
output_schema_table = pd.DataFrame(
    [
        {"field": field, "required": field in output_required, "present_in_actual_response": field in valid_response_payload}
        for field in ["request_id", "model_version", "score", "threshold", "prediction"]
    ]
)

display(schema_table)
display(payload_schema_table)
display(output_schema_table)

,schema,present,qa_use
0,PredictionPayload,True,요청 feature와 request_id 계약
1,PredictionOutput,True,응답 추적 필드 계약
2,HTTPValidationError,True,검증 오류 응답 계약


,field,required,documented_type,training_feature,qa_status
0,request_id,False,string,False,pass
1,heart_rate,True,number,True,pass
2,respiratory_rate,True,number,True,pass
3,body_temperature,True,number,True,pass
4,oxygen_saturation,True,number,True,pass
5,systolic_blood_pressure,True,number,True,pass
6,diastolic_blood_pressure,True,number,True,pass


,field,required,present_in_actual_response
0,request_id,True,True
1,model_version,True,True
2,score,True,True
3,threshold,True,True
4,prediction,True,True


이 출력에서 확인할 핵심은 schema와 실제 응답을 함께 봐야 한다는 점입니다. `PredictionPayload`가 학습 feature를 모두 문서화하고, `PredictionOutput`이 실제 응답 필드와 일치해야 API 계약 증거가 됩니다.

schema에 없는 필드를 client가 임의로 보내거나, schema에 있는 응답 필드를 API가 누락하면 운영 중 원인 분리가 어려워집니다. 이 경우 release gate에서는 API 계약 보완을 owner에게 요청합니다.

## 5. 오류 응답 구조 확인

### 5-1. 422 validation failure가 모델 추론 전 차단되는지 확인

이 셀의 판단은 잘못된 입력이 모델 예측으로 들어가기 전에 검증 오류로 분리되는지 확인하는 것입니다. 이 실패는 모델 metric 저하가 아니라 입력 계약 위반으로 먼저 분류해야 합니다.

오류 응답에는 실패 필드, 오류 유형, 메시지가 남아야 합니다. 그래야 QA가 데이터 owner나 client owner에게 재현 payload와 함께 조치를 요청할 수 있습니다.

이번 invalid payload는 `heart_rate` 타입 오류와 여러 필수 feature 누락을 동시에 포함합니다. 응답이 422이고 detail에 필드별 오류가 있으면 기본 계약은 통과입니다.

In [7]:
invalid_call = call_predict(invalid_payload)
invalid_payload_body = invalid_call["payload"]
error_details = pd.DataFrame(invalid_payload_body.get("detail", []))
if not error_details.empty and "loc" in error_details.columns:
    error_details = error_details.assign(
        field=lambda table: table["loc"].map(lambda loc: loc[-1] if isinstance(loc, list) and loc else None),
        owner_hint=lambda table: table["field"].map(lambda field: "client_or_upstream_payload" if field in FEATURE_COLUMNS else "api_contract"),
    )

error_gate = pd.DataFrame(
    [
        {"gate": "status_422", "observed": invalid_call["status_code"] == 422, "qa_status": "pass" if invalid_call["status_code"] == 422 else "hold"},
        {"gate": "has_detail", "observed": "detail" in invalid_payload_body, "qa_status": "pass" if "detail" in invalid_payload_body else "hold"},
        {"gate": "field_level_errors", "observed": not error_details.empty and "field" in error_details.columns, "qa_status": "pass" if not error_details.empty and "field" in error_details.columns else "review"},
        {"gate": "prediction_not_returned", "observed": "prediction" not in invalid_payload_body, "qa_status": "pass" if "prediction" not in invalid_payload_body else "hold"},
    ]
)

display(pd.DataFrame([{"source": invalid_call["source"], "status_code": invalid_call["status_code"], "fallback_reason": invalid_call["fallback_reason"] or "not_applicable"}]))
display(error_gate)
display(error_details)

,source,status_code,fallback_reason
0,lite_contract_simulation,422,not_applicable


,gate,observed,qa_status
0,status_422,True,pass
1,has_detail,True,pass
2,field_level_errors,True,pass
3,prediction_not_returned,True,pass


,loc,msg,type,field,owner_hint
0,"[body, heart_rate]",Input should be a valid number,float_parsing,heart_rate,client_or_upstream_payload
1,"[body, respiratory_rate]",Field required,missing,respiratory_rate,client_or_upstream_payload
2,"[body, body_temperature]",Field required,missing,body_temperature,client_or_upstream_payload
3,"[body, oxygen_saturation]",Field required,missing,oxygen_saturation,client_or_upstream_payload
4,"[body, systolic_blood_pressure]",Field required,missing,systolic_blood_pressure,client_or_upstream_payload
5,"[body, diastolic_blood_pressure]",Field required,missing,diastolic_blood_pressure,client_or_upstream_payload


이 출력에서 확인할 핵심은 오류가 분리 가능해야 한다는 점입니다. 500 서버 오류로 떨어지면 사용자 입력 문제와 서버 문제를 구분하기 어렵고, 잘못된 입력이 예측 로직까지 들어갈 위험도 의심해야 합니다.

`prediction_not_returned`가 통과하면 검증 실패 요청은 모델 예측 결과로 기록되지 않았다는 뜻입니다. 이런 실패는 4장에서 validation failure 로그와 지표로 관측해야 합니다.

## 6. Deployment artifact 경계와 4장 handoff

### 6-1. Compose API 확인과 Kubernetes manifest 확인 분리

이 셀의 판단은 FastAPI/Compose 확인과 Kubernetes 배포 확인을 섞지 않는 것입니다. 지금까지 확인한 것은 `/predict` 요청/응답 계약입니다. 이 결과만으로 Kubernetes 배포가 성공했다고 말할 수는 없습니다.

Kubernetes 단계에서는 먼저 MLflow tracking service manifest를 확인하고, 그 다음 KServe `InferenceService` manifest가 모델 artifact를 바라보는지 확인합니다. JupyterLite는 Docker daemon이나 Kubernetes cluster를 실행하지 않으므로, 여기서는 GitOps manifest inspection까지만 수행합니다.

따라서 보고서에서는 API 응답 증거와 deployment inspection 증거를 분리해서 씁니다.


In [8]:
container_status = runtime.read_first_json_artifact(
    [
        "artifacts/deployment/chapter_03/container_status_sample.json",
        "artifacts/deployment/chapter_03_container_status_sample.json",
    ]
)

def read_first_text_artifact(relative_paths: list[str]) -> tuple[str, str]:
    for relative_path in relative_paths:
        try:
            return relative_path, runtime.resolve_course_path(relative_path).read_text(encoding="utf-8")
        except FileNotFoundError:
            continue
    raise FileNotFoundError(" | ".join(relative_paths))

mlflow_manifest_path, mlflow_manifest = read_first_text_artifact(
    [
        "demos/ch03_docker_kubernetes/gitops/base/mlflow-tracking.yaml",
        "artifacts/deployment/chapter_03/gitops/base/mlflow-tracking.yaml",
    ]
)
kserve_manifest_path, kserve_manifest = read_first_text_artifact(
    [
        "demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml",
        "artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml",
    ]
)

container_table = pd.DataFrame(
    [
        {"artifact": "container", "check": "execution_mode", "observed": container_status["execution_mode"], "qa_interpretation": "prepared sample 여부를 표시합니다."},
        {"artifact": "container", "check": "status", "observed": container_status["container"]["status"], "qa_interpretation": "container startup 흐름의 예시입니다."},
        {"artifact": "container", "check": "health_status_code", "observed": container_status["health_check"]["status_code"], "qa_interpretation": "prepared health response 예시입니다."},
        {"artifact": "container", "check": "model_version", "observed": container_status["health_check"]["response"].get("model_version"), "qa_interpretation": "서빙 설정의 model_version 노출 예시입니다."},
    ]
)
manifest_table = pd.DataFrame(
    [
        {"artifact": "kubernetes", "check": "mlflow_manifest_path", "observed": mlflow_manifest_path, "qa_interpretation": "MLflow를 Kubernetes에 먼저 배포하는 파일입니다."},
        {"artifact": "kubernetes", "check": "mlflow_deployment", "observed": "kind: Deployment" in mlflow_manifest and "name: mlflow-tracking" in mlflow_manifest, "qa_interpretation": "MLflow tracking 서버 선언 여부입니다."},
        {"artifact": "kubernetes", "check": "mlflow_service", "observed": "kind: Service" in mlflow_manifest and "mlflow-tracking" in mlflow_manifest, "qa_interpretation": "cluster 내부 접근 경로 선언 여부입니다."},
        {"artifact": "kubernetes", "check": "model_artifact_pvc", "observed": "kind: PersistentVolumeClaim" in mlflow_manifest and "ai-quality-models" in mlflow_manifest, "qa_interpretation": "모델 artifact 저장 위치 선언 여부입니다."},
        {"artifact": "kserve", "check": "inferenceservice", "observed": "kind: InferenceService" in kserve_manifest, "qa_interpretation": "모델 serving 후보 선언 여부입니다."},
        {"artifact": "kserve", "check": "storage_uri", "observed": "pvc://ai-quality-models" in kserve_manifest, "qa_interpretation": "KServe가 MLflow/model artifact 저장 위치와 이어지는지 확인합니다."},
    ]
)
deployment_boundary = pd.DataFrame(
    [
        {"evidence": "api_response", "source": valid_call["source"], "can_claim": "요청/응답 계약 확인", "cannot_claim": "container 또는 pod readiness 검증"},
        {"evidence": "container_sample", "source": container_status["execution_mode"], "can_claim": "container smoke 흐름 예시 확인", "cannot_claim": "현재 세션의 live Docker 실행"},
        {"evidence": "gitops_manifest", "source": "repository_files", "can_claim": "MLflow/KServe 배포 선언 확인", "cannot_claim": "Argo CD sync 또는 KServe Ready 성공"},
    ]
)

display(container_table)
display(manifest_table)
display(deployment_boundary)


,artifact,check,observed,qa_interpretation
0,container,execution_mode,prepared_sample,prepared sample 여부를 표시합니다.
1,container,status,running,container startup 흐름의 예시입니다.
2,container,health_status_code,200,prepared health response 예시입니다.
3,container,model_version,baseline-2026-07,서빙 설정의 model_version 노출 예시입니다.


,artifact,check,observed,qa_interpretation
0,kubernetes,mlflow_manifest_path,demos/ch03_docker_kubernetes/gitops/base/mlflo...,MLflow를 Kubernetes에 먼저 배포하는 파일입니다.
1,kubernetes,mlflow_deployment,True,MLflow tracking 서버 선언 여부입니다.
2,kubernetes,mlflow_service,True,cluster 내부 접근 경로 선언 여부입니다.
3,kubernetes,model_artifact_pvc,True,모델 artifact 저장 위치 선언 여부입니다.
4,kserve,inferenceservice,True,모델 serving 후보 선언 여부입니다.
5,kserve,storage_uri,True,KServe가 MLflow/model artifact 저장 위치와 이어지는지 확인합니다.


,evidence,source,can_claim,cannot_claim
0,api_response,lite_contract_simulation,요청/응답 계약 확인,container 또는 pod readiness 검증
1,container_sample,prepared_sample,container smoke 흐름 예시 확인,현재 세션의 live Docker 실행
2,gitops_manifest,repository_files,MLflow/KServe 배포 선언 확인,Argo CD sync 또는 KServe Ready 성공


이 출력에서 확인할 핵심은 prepared deployment artifact의 한계를 보고서에 남기는 것입니다. 파일 inspection은 배포 흐름 이해에는 유효하지만, live smoke test를 대체하지 않습니다.

외부 API가 제공되는 수업에서는 `CH03_API_BASE_URL`을 지정해 live `/predict` 호출 증거를 남기고, Docker/Kubernetes artifact는 배포 설정 확인 증거로만 사용합니다.

### 6-2. API 계약 확인 결과를 observability 확인 항목으로 넘기기

마지막 판단은 API 응답 계약을 4장의 로그와 지표 확인 항목으로 바꾸는 것입니다. 3장에서 `request_id`, `model_version`, `score`, `threshold`, `prediction`이 확인되어야 4장에서 structured log와 prediction distribution을 해석할 수 있습니다.

정상 응답의 추적 필드가 빠지면 4장에서 로그를 보더라도 API 응답과 연결하기 어렵습니다. 오류 응답이 422로 분리되지 않으면 validation failure와 모델 metric 변화를 구분하기 어렵습니다.

이 출력은 4장 observability notebook으로 넘길 handoff입니다.

In [9]:
manifest_checks_passed = bool(manifest_table["observed"].map(bool).all())
serving_packet = pd.DataFrame(
    [
        {
            "evidence": "valid_predict_response",
            "observed": f"source={valid_call['source']}, status={valid_call['status_code']}, model_version={valid_response_payload.get('model_version')}, threshold={valid_response_payload.get('threshold')}",
            "qa_judgment": "정상 응답 추적 필드가 확인되었습니다." if bool(response_gate["qa_status"].isin(["hold"]).sum() == 0) else "정상 응답 계약 보완이 필요합니다.",
            "owner": "QA/Serving",
            "next_action": "4장에서 같은 request_id와 score/prediction 로그를 확인합니다.",
        },
        {
            "evidence": "invalid_payload_response",
            "observed": f"status={invalid_call['status_code']}, error_fields={error_details['field'].dropna().tolist() if 'field' in error_details else []}",
            "qa_judgment": "입력 계약 위반이 422로 분리되었습니다." if invalid_call["status_code"] == 422 else "오류 응답 구조 확인이 필요합니다.",
            "owner": "QA/Client Owner",
            "next_action": "4장에서 validation failure 로그와 실패 필드를 집계합니다.",
        },
        {
            "evidence": "deployment_boundary",
            "observed": f"container={container_status['execution_mode']}, mlflow_kserve_manifest_checks={manifest_checks_passed}",
            "qa_judgment": "manifest는 배포 선언 증거이며 live API 호출 증거와 분리합니다.",
            "owner": "QA/Platform",
            "next_action": "Kubernetes 환경이 제공되면 Argo CD sync와 KServe Ready를 별도로 확인합니다.",
        },
    ]
)
report_sentence = (
    f"/predict 확인은 source={valid_call['source']} 기준으로 수행했으며 status={valid_call['status_code']}, "
    f"model_version={valid_response_payload.get('model_version')}, threshold={valid_response_payload.get('threshold')}, "
    f"prediction={valid_response_payload.get('prediction')}을 확인했습니다. "
    f"잘못된 payload는 status={invalid_call['status_code']}로 분리되었고, "
    f"Kubernetes manifest에서는 MLflow tracking resource와 KServe InferenceService 선언을 확인했습니다. "
    "다만 manifest inspection은 live sync 성공이나 endpoint 응답 성공을 의미하지 않습니다. "
    "4장에서는 같은 필드가 structured log와 prediction distribution에 남는지 확인합니다."
)

display(serving_packet)
print(report_sentence)


,evidence,observed,qa_judgment,owner,next_action
0,valid_predict_response,"source=lite_contract_simulation, status=200, m...",정상 응답 추적 필드가 확인되었습니다.,QA/Serving,4장에서 같은 request_id와 score/prediction 로그를 확인합니다.
1,invalid_payload_response,"status=422, error_fields=['heart_rate', 'respi...",입력 계약 위반이 422로 분리되었습니다.,QA/Client Owner,4장에서 validation failure 로그와 실패 필드를 집계합니다.
2,deployment_boundary,"container=prepared_sample, mlflow_kserve_manif...",manifest는 배포 선언 증거이며 live API 호출 증거와 분리합니다.,QA/Platform,Kubernetes 환경이 제공되면 Argo CD sync와 KServe Ready...


/predict 확인은 source=lite_contract_simulation 기준으로 수행했으며 status=200, model_version=v1, threshold=0.5, prediction=low_risk을 확인했습니다. 잘못된 payload는 status=422로 분리되었고, Kubernetes manifest에서는 MLflow tracking resource와 KServe InferenceService 선언을 확인했습니다. 다만 manifest inspection은 live sync 성공이나 endpoint 응답 성공을 의미하지 않습니다. 4장에서는 같은 필드가 structured log와 prediction distribution에 남는지 확인합니다.


### 6-3. 실패 시 확인 포인트

정상 호출이 실패하면 먼저 payload에 `FEATURE_COLUMNS`가 모두 있고 숫자 타입인지 확인합니다. 필수 입력이 빠지면 모델 점수 문제가 아니라 API 입력 계약 문제입니다.

외부 API 호출이 실패하고 fallback으로 전환되면 보고서에 live API를 검증했다고 쓰지 않습니다. 외부 API URL, 네트워크 접근, CORS 또는 인증 조건을 확인한 뒤 live smoke test를 별도로 수행해야 합니다.

오류 응답이 422가 아니면 실패를 분리하기 어렵습니다. release gate에서는 오류 상태 코드, 실패 필드, owner, 재현 payload를 함께 남기고 운영 승인 판단을 보류합니다.

prepared Docker/Kubernetes artifact는 배포 설정 확인용입니다. live container, Pod readiness, Service 호출 성공은 별도 로컬 Demo 또는 외부 환경에서 확인해야 합니다.